# Chapter 13: Retrieval-Augmented Generation (RAG)
## Notebook 01 — RAG Fundamentals

This notebook introduces the core building blocks of RAG: **why** we need it, how **embeddings + cosine similarity** drive retrieval, how to build a tiny **vector store from scratch**, and how to put together your **first end-to-end RAG** answer with a mock LLM.

### What you'll learn

| Topic | Section |
|-------|--------|
| Why RAG: hallucination, recency, private data, context limits | §1 |
| Embeddings recap and cosine similarity | §2 |
| Build an in-memory vector store from NumPy | §3 |
| Naive retrieval: encode query, top-k by cosine | §4 |
| First end-to-end RAG with a mock LLM | §5 |
| Evaluation: hit@k, MRR, precision@k | §6 |

**Estimated time:** 2.5–3 hours

---
*Generated by Berta AI*

---
## 1. Why RAG?

Out-of-the-box, large language models have four well-known weaknesses that RAG directly attacks:

1. **Hallucination** — fluent, confident, but factually wrong output.
2. **Stale knowledge** — the model only knows what was in its training data.
3. **No access to private data** — your wiki, tickets, and product docs.
4. **Context-window limits** — you cannot just paste an entire corpus into every prompt.

**RAG** fixes all four by retrieving the *most relevant* passages from your corpus at query time and injecting them into the prompt. The model then answers with evidence in front of it.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

print("Setup complete. Working dir:", os.getcwd())

### A motivating example

Suppose a user asks *"What is HyDE in retrieval?"*. A vanilla LLM might invent an answer. With RAG, we look up our corpus, find the passage that defines HyDE, and feed it to the model. The answer is now grounded — and *citable*.

In [ ]:
# Tiny demo corpus
corpus = {
    "p1": "RAG combines a retriever with a generator. The retriever finds relevant passages.",
    "p2": "HyDE stands for Hypothetical Document Embeddings. The retriever first asks the LLM to draft a hypothetical answer, then embeds that draft and uses it as the search vector.",
    "p3": "BM25 is a sparse retriever based on term frequency and inverse document frequency.",
    "p4": "Cosine similarity between two vectors equals their dot product divided by the product of their L2 norms.",
}
for k, v in corpus.items():
    print(f"[{k}] {v}")

---
## 2. Embeddings and Cosine Similarity

An **embedding** is a dense numeric vector that represents a piece of text. Good embeddings place semantically similar texts near each other in vector space.

**Cosine similarity** is the angle between two vectors:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}$$

When both vectors are L2-normalized, cosine similarity reduces to a single dot product. We will exploit that throughout the chapter.

In [ ]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

# Sanity checks
v1 = np.array([1.0, 0.0])
v2 = np.array([1.0, 0.0])
v3 = np.array([0.0, 1.0])
v4 = np.array([-1.0, 0.0])
print("identical:", cosine_similarity(v1, v2))   # 1.0
print("orthogonal:", cosine_similarity(v1, v3))  # 0.0
print("opposite:", cosine_similarity(v1, v4))    # -1.0

For text we cannot use raw words as vectors. The simplest realistic embedding is **TF-IDF** — every document becomes a sparse vector of term weights. We will use scikit-learn for that.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cos

texts = list(corpus.values())
ids = list(corpus.keys())

vec = TfidfVectorizer().fit(texts)
M = vec.transform(texts)
print("TF-IDF matrix shape:", M.shape)

query = "What is HyDE?"
q = vec.transform([query])
sims = sk_cos(q, M).ravel()
for i in np.argsort(-sims):
    print(f"{ids[i]:>3}  sim={sims[i]:.3f}  | {texts[i][:70]}")

---
## 3. A Vector Store From Scratch

A vector store is just an array of embeddings plus the ability to query them. Here is one in ~25 lines of NumPy.

In [ ]:
class TinyVectorStore:
    def __init__(self):
        self.vectors = []
        self.ids = []
        self.texts = []

    def add(self, ids, texts, vectors):
        for i, t, v in zip(ids, texts, vectors):
            self.ids.append(i)
            self.texts.append(t)
            self.vectors.append(np.asarray(v, dtype=np.float32))

    def search(self, query_vector, top_k=3):
        if not self.vectors:
            return []
        q = np.asarray(query_vector, dtype=np.float32)
        mat = np.vstack(self.vectors)
        # Normalize to compute cosine via dot product
        mat_n = mat / np.maximum(np.linalg.norm(mat, axis=1, keepdims=True), 1e-12)
        q_n = q / max(np.linalg.norm(q), 1e-12)
        scores = mat_n @ q_n
        order = np.argsort(-scores)[:top_k]
        return [(self.ids[i], float(scores[i]), self.texts[i]) for i in order]

# Build a store from our TF-IDF vectors (densified for simplicity)
store = TinyVectorStore()
store.add(ids, texts, M.toarray())
print("Indexed", len(store.vectors), "documents.")

---
## 4. Naive Retrieval

Encoding the query and asking the store for the top-k matches is now one line of code.

In [ ]:
def retrieve(query, store, vectorizer, top_k=3):
    q = vectorizer.transform([query]).toarray()[0]
    return store.search(q, top_k=top_k)

for q in ["What is HyDE?", "How does BM25 work?", "Define cosine similarity."]:
    print(f"\nQuery: {q}")
    for cid, score, text in retrieve(q, store, vec, top_k=2):
        print(f"  {cid}  score={score:.3f}  {text[:70]}")

---
## 5. First End-to-End RAG

We now have everything we need. The recipe is:

1. **Encode** the query
2. **Retrieve** top-k relevant chunks
3. **Build a prompt** that includes both the question and the retrieved chunks
4. **Generate** an answer (we use a mock LLM that templates the chunks deterministically)

This means **no API keys are required** — but the same `RAGPipeline` class works with a real OpenAI or Anthropic client by swapping the `llm_client` argument.

In [ ]:
from rag_pipeline import RAGPipeline

documents = {f"d{i+1}": txt for i, txt in enumerate(texts)}

pipe = RAGPipeline()
n_chunks = pipe.index_documents(documents)
print("Indexed", n_chunks, "chunks")

response = pipe.answer("What is HyDE in retrieval?")
print("\nAnswer:")
print(response.answer)
print("\nRetrieved contexts:")
for c in response.contexts:
    print(f"  [{c.chunk_id}] score={c.score:.3f}  {c.text[:80]}")

---
## 6. Evaluation: hit@k, MRR, Precision@k

How do we know retrieval is working? A handful of standard metrics with binary relevance labels:

- **hit@k** — 1 if any of the top-k results is relevant, else 0
- **Mean Reciprocal Rank (MRR)** — average of `1 / rank_of_first_relevant`
- **precision@k** — fraction of top-k that are relevant

In [ ]:
def hit_at_k(retrieved_ids, gold_ids, k):
    return int(any(rid in set(gold_ids) for rid in retrieved_ids[:k]))

def reciprocal_rank(retrieved_ids, gold_ids):
    gold = set(gold_ids)
    for rank, rid in enumerate(retrieved_ids, start=1):
        if rid in gold:
            return 1.0 / rank
    return 0.0

def precision_at_k(retrieved_ids, gold_ids, k):
    if k == 0:
        return 0.0
    gold = set(gold_ids)
    return sum(1 for rid in retrieved_ids[:k] if rid in gold) / k

# Toy benchmark with three queries
gold = {
    "What is HyDE in retrieval?": ["d2"],
    "How does BM25 work?": ["d3"],
    "Define cosine similarity.": ["d4"],
}

for q, gids in gold.items():
    hits = pipe.retrieve(q, top_k=3)
    rids = [h.chunk_id.split("::")[0] for h in hits]
    print(f"{q}\n  retrieved: {rids}\n  hit@3 = {hit_at_k(rids, gids, 3)}, "
          f"RR = {reciprocal_rank(rids, gids):.2f}, "
          f"P@3 = {precision_at_k(rids, gids, 3):.2f}")

---
## 7. Key Takeaways

- **RAG = retrieve + generate.** Retrieval grounds the LLM in evidence the model never saw or has forgotten.
- **Cosine similarity over embeddings** is the workhorse of retrieval. Normalize once and dot products do the rest.
- **A vector store is just an array of vectors** with a top-k query operation — easy to build from NumPy for learning.
- **hit@k and MRR** tell you whether your retriever is finding the right passages; both are easy to compute with binary labels.

Next up: **Notebook 02** — chunking strategies, embedding model choice, real vector stores, reranking, and prompt assembly with citations.

---
*Generated by Berta AI*